# ⚡ Eco Video Upscaler & Subtitle AI — Cloud Edition

Giao diện WebUI 2-trong-1 chạy trên **GPU Google Colab (Tesla T4 15GB / L4 24GB / A100)**:

1. **⚡ Tab Phục chế & Siêu phân giải (Upscale AI)**:
   * Hỗ trợ **SeedVR2-3B (ByteDance DiT Video SOTA)**, **CCSR v2 (Diffusion)**, và **NomosUni / RealESRGAN**.
   * Khử flicker, phục hồi chi tiết chân dung tự nhiên, bảo toàn 100% số khung hình và âm thanh gốc.

2. **📝 Tab Phụ đề tự động (Whisper + Gemini)**:
   * Tự động nhận diện giọng nói đa ngôn ngữ (Anh, Trung, Nhật, Hàn, Việt...).
   * Dịch ngữ cảnh tự nhiên sang tiếng Việt bằng **Google Gemini**.
   * Xuất `.srt`, `.ass` và ghép cứng (burn-in) phụ đề vào video bằng FFmpeg `libass` (phông Noto Sans).


In [ ]:
# [BƯỚC 1] Kiểm tra GPU (Khuyên dùng GPU T4)
!nvidia-smi

In [ ]:
# [BƯỚC 2] Tải mã nguồn dự án và cài đặt thư viện
%cd /content
!rm -rf /content/upscale_video
!git clone https://github.com/mloc07/upscale_video.git
%cd /content/upscale_video

# 1. Thư viện cho Upscale Video AI
!pip install -q gradio onnxruntime-gpu diffusers accelerate transformers nvidia-ml-py torchvision huggingface_hub rich opencv-python mediapy einops omegaconf timm safetensors rotary-embedding-torch

# 2. Thư viện cho Phụ đề tự động (Whisper + Gemini + CUDA 12 + Phông chữ)
!pip install -q -e ".[subtitle]"
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12
!apt-get install -qq -y fonts-noto-core fonts-noto-cjk fontconfig


In [ ]:
# [BƯỚC 2B] (Tùy chọn) Cài đặt API Key cho tính năng Phụ đề (Gemini API)
# Cách 1: Thêm vào mục Secrets (biểu tượng chìa khóa ở thanh bên trái Colab) với tên GEMINI_API_KEY
# Cách 2: Điền trực tiếp API Key vào bên dưới nếu chưa dùng Secrets
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("✅ Đã nạp thành công GEMINI_API_KEY từ Colab Secrets!")
except Exception:
    # Bỏ dấu comment và điền API key nếu bạn không dùng Secrets:
    # os.environ["GEMINI_API_KEY"] = "AIzaSy..."
    print("💡 Nếu sử dụng tab Dịch phụ đề Gemini, hãy thêm GEMINI_API_KEY vào Colab Secrets hoặc gán trực tiếp.")


In [ ]:
# [BƯỚC 3A] Tải các mô hình AI tiêu chuẩn (Nhanh 30–50 FPS)
!python scripts/download_model.py 2xNomosUni_compact_otf_medium
!python scripts/download_model.py 2x-realesrgan-x2plus
!python scripts/download_model.py realesr-general-x4v3
!python scripts/download_model.py realesr-animevideov3
!python scripts/download_model.py 2x-spanx2-ch48

In [ ]:
# [BƯỚC 3B] (Tùy chọn) Tải mã nguồn và trọng số mô hình khuếch tán CCSR v2
# Chạy ô này nếu bạn muốn sử dụng mô hình CCSR v2 (Diffusion Model)
!git clone -b CCSR-v2.0 https://github.com/csslc/CCSR.git ccsr_core
!python scripts/download_ccsr.py

In [ ]:
# [BƯỚC 3C] (Khuyên dùng) Tải mã nguồn và trọng số ByteDance SeedVR2-3B
# Chạy ô này nếu bạn muốn sử dụng mô hình tân tiến SeedVR2-3B (ByteDance DiT Video SOTA)
!git clone https://github.com/ByteDance/SeedVR.git seedvr_core
!python scripts/download_seedvr2.py
!python scripts/patch_seedvr_colab.py seedvr_core


In [ ]:
# [BƯỚC 4] Khởi động giao diện Web (Gồm 2 Tab: Upscale AI & Phụ đề tự động)
%cd /content/upscale_video
!git pull origin main
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True && python colab_app.py --share
